### Tools

Built-in Tools

In [19]:
import os
cache_dir = 'D:/Development/ML/Deep Learning/GenAI/.hf_cache'
os.environ['HF_HOME'] = cache_dir
os.environ['TRANSFORMERS_CACHE'] = cache_dir
os.environ['HF_DATASETS_CACHE'] = cache_dir
os.makedirs(cache_dir, exist_ok=True)

from langchain_community.tools import DuckDuckGoSearchRun
from langchain_huggingface import ChatHuggingFace, HuggingFacePipeline, HuggingFaceEmbeddings, HuggingFaceEndpoint
from langchain_core.tools import tool
from langchain_core.tools import StructuredTool, BaseTool
from pydantic import BaseModel, Field
from typing import Type

from transformers import pipeline, AutoTokenizer, AutoModelForCausalLM

In [20]:
def search_duckduckgo(query: str) -> str:
    """Searches DuckDuckGo for the given query and returns the results."""
    search_tool = DuckDuckGoSearchRun()
    return search_tool.invoke(query)

In [21]:
search_duckduckgo("What is the capital of France?")

'France(/ˈfræns/ⓘor /ˈfrɑːns/; French pronunciation: [fʁɑ̃s]), officially the French Republic (French: République française, French pronunciation: [ʁepyblik fʁɑ̃sɛz]), is a country in Western Europe. It also includes various departments and territories ofFranceoverseas. MainlandFranceextends from the Mediterranean Sea to the English Channel and the North Sea, and from ... Where in the World is Paris found? Paris is thecapitalofFrance(French Republic), situated in the Western Europe subregion of Europe. In Paris, the currency used is Euro (€), which is the official currency used inFrance.TheLatitude, Longitude cordinates of Paris are 48.8534, 2.3488. Paris, thecapitaland largest city ofFrance,issituated in the north-central region of the country along the Seine River.) The city has a rich history, with evidence of human habitation dating back to approximately 7600 BCE.) Over the centuries, Paris has evolved into a major center of commerce, culture, and politics, playing a pivotal role i

Custom Tools

In [22]:
@tool("multiply")
def multiply(a: int, b: int) -> int:
    """Multiplies two numbers."""
    return a * b

In [23]:
multiply.invoke({"a": 5, "b": 3})

15

In [25]:
print(multiply.name)
print(multiply.description)
print(multiply.args)

multiply
Multiplies two numbers.
{'a': {'title': 'A', 'type': 'integer'}, 'b': {'title': 'B', 'type': 'integer'}}


In [26]:
# What Model sees when it calls the tool
multiply.args_schema.model_json_schema()

{'description': 'Multiplies two numbers.',
 'properties': {'a': {'title': 'A', 'type': 'integer'},
  'b': {'title': 'B', 'type': 'integer'}},
 'required': ['a', 'b'],
 'title': 'multiply',
 'type': 'object'}

Building Tool using Structured Tool (Use of Pydantic Model)

In [27]:
class MultiplyInterface(BaseModel): # Pydantic model for structured tool
    a: int = Field(description="The first number to multiply")
    b: int = Field(description="The second number to multiply")

In [28]:
def multiply_structured(a: int, b: int) -> int:
    return a * b

multiply_structured_tool = StructuredTool.from_function(
    func=multiply_structured,
    name="multiply_structured",
    description="Multiplies two numbers using a structured tool interface.",
    args_schema=MultiplyInterface
)

In [29]:
print(multiply_structured_tool.invoke({"a": 7, "b": 4}))
print(multiply_structured_tool.args_schema.model_json_schema())

28
{'properties': {'a': {'description': 'The first number to multiply', 'title': 'A', 'type': 'integer'}, 'b': {'description': 'The second number to multiply', 'title': 'B', 'type': 'integer'}}, 'required': ['a', 'b'], 'title': 'MultiplyInterface', 'type': 'object'}


Building Tool using BaseTool (The Parent Class for all Tools)

In [30]:
class MultipleBaseTool(BaseTool):
    name: str = "multiply_base_tool"
    description: str = "Multiplies two numbers using a BaseTool interface."
    args_schema: Type[BaseModel] = MultiplyInterface # using the same Pydantic model for args schema

    def _run(self, a: int, b: int) -> int:
        return a * b

In [31]:
multiply_tool = MultipleBaseTool()
print(multiply_tool.invoke({"a": 6, "b": 9}))
print(multiply_tool.args_schema.model_json_schema())

54
{'properties': {'a': {'description': 'The first number to multiply', 'title': 'A', 'type': 'integer'}, 'b': {'description': 'The second number to multiply', 'title': 'B', 'type': 'integer'}}, 'required': ['a', 'b'], 'title': 'MultiplyInterface', 'type': 'object'}


Tool Kit

In [32]:
class MathTookit:
    def __init__(self):
        self.multiply = MultipleBaseTool()
        
    def get_tools(self):
        return [self.multiply]

Binding Tool with LLM

In [33]:
def temperature_tool(location: str) -> float:
    """
        Returns the current temperature for a given location.
        Args:
            location: The location to get the temperature for. (string)
    """
    return 22.5

In [34]:
model_id = "google/gemma-3-1b-it"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id, device_map="auto", dtype="auto")
messages = [
    {"role": "system", "content": "For weather use the available tool to get the current temperature."},
    {"role": "user", "content": "What is the current temperature?"}
]

prompt = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    tools=[temperature_tool],
    return_dict=True,
    return_tensors="pt"
)

In [ ]:
res = model.generate(**prompt.to(model.device), max_new_tokens=500, temperature=0.2)

In [ ]:
output = tokenizer.decode(res[0])
print(output)